# API-Football : matchs (fixtures) → bronze

**Rôle** : charger les fichiers parquet déposés par `api_football_fixtures_l_loader` dans la table Delta bronze.

| | |
|---|---|
| **Source** | `/Volumes/<CATALOG>/landing/files/api_football/fixtures/<aaaa>/<mm>/<jj>/*.parquet` |
| **Cible** | `<CATALOG>.bronze.api_football_fixtures` |
| **Méthode** | `COPY INTO` (SQL) : chaque fichier de la landing est chargé **une seule fois** ; un fichier déjà chargé est ignoré |
| **Colonnes** | **toutes** les colonnes déduites par dlthub + 3 colonnes techniques |
| **Doublons** | **tous les dépôts sont conservés** (un même match peut apparaître plusieurs fois) ; la dernière version par match sera retrouvée en **silver** |
| **Coût** | une requête courte ; ne relit jamais un fichier déjà chargé |

**Colonnes techniques ajoutées**

| Colonne | Contenu | Origine |
|---|---|---|
| `_date_depot` | date et heure du dépôt dans la landing | `_dlt_load_id` (horodatage Unix) converti en date |
| `_fichier_source` | chemin complet du fichier parquet lu | métadonnée `_metadata.file_path` fournie par Databricks |
| `_date_chargement_bronze` | date et heure d'écriture en bronze | `current_timestamp()` au moment du `COPY INTO` |

## 1. Paramètre

| Paramètre | Valeur en dev | Sert à… |
|---|---|---|
| `CATALOG` | `dev` | indiquer dans quel catalog lire la landing et écrire la bronze (`prod` en production) |

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")

## 2. Configuration

`SOURCE` et `TABLE` sont des constantes de **ce** loader (identiques en dev et en prod).
Le nom de la table bronze suit la convention `<source>_<table>`.

In [0]:
CATALOG = dbutils.widgets.get("CATALOG").strip()

SOURCE = "api_football"
TABLE  = "fixtures"

CHEMIN_LANDING = f"/Volumes/{CATALOG}/landing/files/{SOURCE}/{TABLE}/"
TABLE_BRONZE   = f"{CATALOG}.bronze.{SOURCE}_{TABLE}"

print("Source :", CHEMIN_LANDING)
print("Cible  :", TABLE_BRONZE)

## 3. Création de la table (si elle n'existe pas)

Table **vide et sans colonnes** : c'est `COPY INTO` qui déduira les colonnes des fichiers parquet au premier chargement (`mergeSchema`).
`IF NOT EXISTS` : la cellule est **rejouable** sans risque.

ℹ️ Propriétaire de la table = l'identité qui l'a créée (Cyrille G en dev ; `sp-data-prod` en prod via le job).

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_BRONZE}
    COMMENT 'Bronze API-Football matchs : tous les dépôts de la landing, colonnes dlthub + _date_depot, _fichier_source, _date_chargement_bronze'
""")
print("✅ Table prête :", TABLE_BRONZE)

## 4. État avant chargement

Nombre de fichiers présents dans la landing et nombre de lignes déjà en bronze.

In [0]:
import os

fichiers_landing = [
    os.path.join(dossier, f)
    for dossier, _, fichiers in os.walk(CHEMIN_LANDING)
    for f in fichiers if f.endswith(".parquet")
]
# Au tout premier lancement, la table existe mais n'a encore aucune colonne :
# Databricks refuse alors de la lire (DELTA_READ_TABLE_WITHOUT_COLUMNS) → on compte 0 ligne.
try:
    lignes_avant = spark.table(TABLE_BRONZE).count()
except Exception as e:
    if "DELTA_READ_TABLE_WITHOUT_COLUMNS" in str(e):
        lignes_avant = 0
    else:
        raise

print(f"📂 Fichiers parquet dans la landing : {len(fichiers_landing)}")
print(f"📊 Lignes en bronze avant chargement : {lignes_avant}")

## 5. Chargement : `COPY INTO`

| Partie | Rôle |
|---|---|
| `SELECT *, …` | toutes les colonnes de dlthub + les 3 colonnes techniques |
| `timestamp_seconds(CAST(_dlt_load_id AS DOUBLE))` | convertit l'identifiant de chargement dlthub (secondes depuis le 01/01/1970) en date et heure |
| `_metadata.file_path` | chemin du fichier lu, fourni par Databricks |
| `PATTERN = '*/*/*/*.parquet'` | lit les fichiers rangés en `<aaaa>/<mm>/<jj>/` |
| `FORMAT_OPTIONS ('mergeSchema' = 'true')` | réunit les colonnes de **tous** les fichiers lus (un fichier peut avoir une colonne que l'autre n'a pas) |
| `COPY_OPTIONS ('mergeSchema' = 'true')` | ajoute à la table une **nouvelle colonne** si un fichier en apporte une |

Résultat affiché : `num_affected_rows` / `num_inserted_rows` = lignes ajoutées ; **0** si aucun nouveau fichier.

In [0]:
requete_copy = f"""
COPY INTO {TABLE_BRONZE}
FROM (
  SELECT *,
         timestamp_seconds(CAST(_dlt_load_id AS DOUBLE)) AS _date_depot,
         _metadata.file_path                             AS _fichier_source,
         current_timestamp()                             AS _date_chargement_bronze
  FROM '{CHEMIN_LANDING}'
)
FILEFORMAT = PARQUET
PATTERN = '*/*/*/*.parquet'
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
"""
print(requete_copy)

resultat = spark.sql(requete_copy)
display(resultat)

## 6. Vérifications

1. Lignes ajoutées et total.
2. Répartition par **dépôt** et par **fichier source** (chaque fichier de la landing doit apparaître une seule fois).
3. Doublons : nombre de lignes vs nombre de matchs distincts (`fixture__id`). Des doublons sont **attendus** si un même match a été déposé plusieurs fois.

In [0]:
from pyspark.sql import functions as F

df = spark.table(TABLE_BRONZE)
lignes_apres = df.count()
print(f"📊 Lignes en bronze : {lignes_avant} → {lignes_apres} (+{lignes_apres - lignes_avant})")
print(f"🧱 Colonnes : {len(df.columns)}")

display(df.groupBy("_date_depot", "_fichier_source")
          .agg(F.count("*").alias("nb_lignes"))
          .orderBy("_date_depot"))

In [0]:
nb_matchs = df.select("fixture__id").distinct().count()
print(f"🔁 Lignes : {lignes_apres} | matchs distincts (fixture__id) : {nb_matchs} | doublons : {lignes_apres - nb_matchs}")

display(df.groupBy("league__id", "league__name", "league__season")
          .agg(F.count("*").alias("nb_lignes"))
          .orderBy("league__id", "league__season"))

## 7. Historique de la table

Chaque `COPY INTO` est enregistré dans l'historique Delta (date, opération, nombre de fichiers et de lignes).

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TABLE_BRONZE}")
          .select("version", "timestamp", "userName", "operation", "operationMetrics")
          .limit(5))